In [3]:
import numpy as np
import pandas as pd
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM, Bidirectional,
    Dense, Dropout, RepeatVector, GlobalAveragePooling1D
)

dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

X = df["text"].astype(str).values
y = df["label"].astype(int).values


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


max_words = 10000
seq_len = 50

vectorizer = TextVectorization(
    max_tokens=max_words,
    output_sequence_length=seq_len
)

vectorizer.adapt(X_train)

all_predictions = {}

def train_and_evaluate(model, name):
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )
    
    y_pred = np.argmax(model.predict(X_test), axis=1)
    acc = accuracy_score(y_test, y_pred)

    all_predictions[name] = y_pred
    
    print(f"\n{name} accuracy: {acc:.4f}")
    return acc


# LSTM
lstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Bi-LSTM
bilstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# ED-LSTM (uproszczony)
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    LSTM(64),
    RepeatVector(seq_len),
    LSTM(64, return_sequences=True),
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(3, activation="softmax")
])


results = {}

results["LSTM"] = train_and_evaluate(lstm_model, "LSTM")
results["Bi-LSTM"] = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["ED-LSTM"] = train_and_evaluate(ed_lstm_model, "ED-LSTM")


results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
print("\n=== Overall Results ===")
print(results_df)


label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names
    ))

    cm = confusion_matrix(y_test, y_pred)

    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly predicted": cm.diagonal(),
        "Total in test set": cm.sum(axis=1),
        "Accuracy per class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed class summary:")
    print(summary)

Epoch 1/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 36s 62ms/step - accuracy: 0.4481 - loss: 1.0117 - val_accuracy: 0.4989 - val_loss: 0.9440
Epoch 2/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.4958 - loss: 0.9131 - val_accuracy: 0.5770 - val_loss: 0.8550
Epoch 3/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 38s 57ms/step - accuracy: 0.6486 - loss: 0.7675 - val_accuracy: 0.6192 - val_loss: 0.8165
286/286 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step

LSTM accuracy: 0.6274
Epoch 1/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 61s 106ms/step - accuracy: 0.5913 - loss: 0.8547 - val_accuracy: 0.6441 - val_loss: 0.7646
Epoch 2/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 29s 56ms/step - accuracy: 0.7073 - loss: 0.6617 - val_accuracy: 0.6523 - val_loss: 0.7746
Epoch 3/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 30s 58ms/step - accuracy: 0.7536 - loss: 0.5738 - val_accuracy: 0.6468 - val_loss: 0.8185
286/286 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step

Bi-LSTM accuracy: 0.6502
Epoch 1/3
514/514 ━━━━━━━━━━━━━━━━━━━━ 52s 90ms/step - accuracy: 0.4467 - loss: 1.0191 - val

C:\Users\YOGA\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\YOGA\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\YOGA\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [5]:
import numpy as np
import pandas as pd
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM, Bidirectional,
    Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

# Dane
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

X = df["text"].astype(str).values
y = df["label"].astype(int).values

# Train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Wektoryzacja tekstu
max_words = 10000
seq_len = 50

vectorizer = TextVectorization(
    max_tokens=max_words,
    output_sequence_length=seq_len
)

vectorizer.adapt(X_train)

# Early stopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

all_predictions = {}

def train_and_evaluate(model, name):
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=1
    )

    y_pred = np.argmax(model.predict(X_test), axis=1)
    acc = accuracy_score(y_test, y_pred)

    all_predictions[name] = y_pred

    print(f"\n{name} accuracy: {acc:.4f}")
    return acc

# LSTM
lstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Bi-LSTM
bilstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# ED-LSTM, uproszczony wariant do klasyfikacji
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(max_words, 64),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

# Trening
results = {}

results["LSTM"] = train_and_evaluate(lstm_model, "LSTM")
results["Bi-LSTM"] = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["ED-LSTM"] = train_and_evaluate(ed_lstm_model, "ED-LSTM")

# Wyniki ogólne
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
print("\n=== Overall Results ===")
print(results_df)

# Szczegółowe wyniki
label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)

    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly predicted": cm.diagonal(),
        "Total in test set": cm.sum(axis=1),
        "Accuracy per class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed class summary:")
    print(summary)

Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 30s 51ms/step - accuracy: 0.4498 - loss: 1.0191 - val_accuracy: 0.4507 - val_loss: 1.0145
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 51s 99ms/step - accuracy: 0.4734 - loss: 0.9405 - val_accuracy: 0.5537 - val_loss: 0.8771
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 26s 50ms/step - accuracy: 0.6425 - loss: 0.7813 - val_accuracy: 0.6241 - val_loss: 0.8049
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 27s 52ms/step - accuracy: 0.7163 - loss: 0.6706 - val_accuracy: 0.6227 - val_loss: 0.8068
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 26s 51ms/step - accuracy: 0.7536 - loss: 0.6009 - val_accuracy: 0.6162 - val_loss: 0.8309
286/286 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step

LSTM accuracy: 0.6296
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 55s 95ms/step - accuracy: 0.5865 - loss: 0.8562 - val_accuracy: 0.6430 - val_loss: 0.7818
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 47s 92ms/step - accuracy: 0.7051 - loss: 0.6645 - val_accuracy: 0.6378 - val_loss: 0.7770
Epoch 3/10
514/514 ━━━━━━